In [ ]:
import math
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from itertools import count
from importnb import Notebook

with Notebook():
    from LabLatencyModel import LatencyModel, MultiDULatencyModel
    from LabCacheEngine import CacheEngineEnv
    from LabUserTileRequest import UserTileRequestEvents

In [ ]:
class EnvWrapper(gym.Env):
    """
    Simple wrapper that delegates all calls to an inner env.
    Subclass this to create your own wrappers.
    """

    def __init__(
        self, 
        n: int,
        m: int,
        n_layers: int,
        lam: float,
        theta: float,
        users_env: None,
        du_caches: None,
        mec_cache: None,
        latency_model: None 
    ):
        super().__init__()

        self.step_count = 0

        self.n = n  # number of tiles per row/column
        self.m = m  # number of tiles per row/column
        self.n_layers = n_layers  # number of layers (base + enhancement)
        
        self.gain_if_prefetched = 1.0
        self.loss_if_not_prefetched = -1.0

        self.theta = theta
        self.lam = lam

        self.users_env = users_env
        self.du_caches = du_caches
        self.mec_cache = mec_cache
        self.latency_model = latency_model

    def sample_action(self):
        n_users = self.users_env.n_users
        
        tiles = np.zeros(self.n * self.m, dtype=int)
        c = self.n // 2
        if self.n % 2 == 1:
            center_idx = c * self.n + c
            tiles[center_idx] = 1
        else:
            centers = [(c-1, c-1), (c-1, c), (c, c-1), (c, c)]
            for x, y in centers:
                tiles[y * self.n + x] = 1

        return {
            'video': np.random.randint(0, self.users_env.n_videos),
            'gop': np.random.randint(0, self.users_env.n_gops),
            'tiles': tiles.tolist()
        }

    def step(self, action):
        reward = 0.0
        info = {}

        ### STEP 1: prefetching segments in the cache based on action taken ###
        du_bitmaps = [ 
            cache.prefetching(action) for cache in self.du_caches
        ] if self.du_caches and len(self.du_caches) > 0 else None
        mec_bitmap = self.mec_cache.prefetching(action) if self.mec_cache else None

        ### STEP 2: Generate user requests ###
        req, req_t = self.users_env.step(du_bitmaps, mec_bitmap)
        info['request_time'] = req_t
        info['user_request'] = req

        ### STEP 2.1: Compute Content Popularity Distribution ###
        for cache in self.du_caches:
            cache.update_content_popularity(req)
        self.mec_cache.update_content_popularity(req)

        info['content_popularity'] = [
            cache.get_content_popularity() for cache in self.du_caches
        ] + [self.mec_cache.get_content_popularity()]

        ### STEP 3: compute latencies for each request ###
        p = req['p']
        u = req['u']
        tiles = req['tiles']

        users_per_du = self.users_env.n_users // self.users_env.n_nodes
        req_latency, tile_latencies = self.latency_model.total_request_latency_multi_du(
            p,
            u,
            tiles,
            users_per_du,
            self.users_env.n_users
        )
        tiles_received = tile_latencies

        ### STEP 4: Calculate metrics: cache hits and misses, and reward ###
        cache_stats = self.compute_cache_stats([req])
        info.update(cache_stats)

        cache_current_capacity = sum(cache.get_current_capacity() for cache in self.du_caches) if self.du_caches else 0
        cache_current_capacity += self.mec_cache.get_current_capacity() if self.mec_cache else 0

        cache_total_capacity = sum(cache.max_capacity for cache in self.du_caches) if self.du_caches else 0
        cache_total_capacity += self.mec_cache.max_capacity if self.mec_cache else 0

        ### STEP 4.1: Compute QoE-based requests completed ###
        u = req["u"]
        tiles = req["tiles"]
        tile_latencies = tiles_received
        tile_latency_dict = {
            (tid, layer): lat for (tid, layer, lat) in tile_latencies
        }

        psnr_sum = 0.0
        sum_tiles = 0
        for tile in tiles:
            tid = tile["tile"]
            layer = tile["layer"]
            
            psnr = 0.0    
            if layer == 1:
                if (tid, 0) in tile_latency_dict:
                    psnr += 30.0
                if (tid, 1) in tile_latency_dict:
                    psnr += 10.0
                
                sum_tiles += 1
            psnr_sum += psnr
        
        avg_psnr = psnr_sum / sum_tiles if sum_tiles else 0.0

        info['average_psnr'] = avg_psnr
        info['cache_current_capacity'] = cache_current_capacity
        info['cache_total_capacity'] = cache_total_capacity
        info['cache_utilization'] = (cache_current_capacity / cache_total_capacity) if cache_total_capacity else 0.0

        # compute reward for this video's action and add
        total_requests = len(req['tiles'])
        
        # Compute current_p: probability for each (vid, tile_idx) based on requests
        tile_counts = {}
        vid = req["video"]
        tiles = req["tiles"]
        for tile in tiles:
            key = (vid, tile['layer'], tile['tile'])
            tile_counts[key] = tile_counts.get(key, 0) + 1

        # Normalize by total_requests to get probabilities
        current_p = {
            key: count / total_requests
            for key, count in tile_counts.items()
        }
    
        reward += self.compute_expected_cpt_reward(
            current_p=current_p,
            requests=[req]
        )

        done = self.users_env.all_users_done()

        self.step_count += 1

        return {}, reward, done, info

    def compute_cache_stats(self, reqs):
        enh_layer_cache_hits = 0
        base_layer_cache_hits = 0
        enh_layer_cache_misses = 0
        base_layer_cache_misses = 0
        
        for req in reqs:
            tiles = req['tiles']
            for tile in tiles:
                layer = tile['layer']
                # hit = tile['events']['alpha_p_u']
                hit = tile['events']['alpha_M_u']  # Considering MEC cache hits

                if layer == 0 and hit == 1:
                    base_layer_cache_hits += 1
                elif layer == 1 and hit == 1:
                    enh_layer_cache_hits += 1
                elif layer == 0 and hit == 0:
                    base_layer_cache_misses += 1
                elif layer == 1 and hit == 0:
                    enh_layer_cache_misses += 1

        return {
            'base_layer_cache_hits': int(base_layer_cache_hits),
            'enh_layer_cache_hits': int(enh_layer_cache_hits),
            'base_layer_cache_misses': int(base_layer_cache_misses),
            'enh_layer_cache_misses': int(enh_layer_cache_misses)
        }

    def compute_expected_cpt_reward(
        self, 
        current_p, 
        requests
    ) -> float:
        expected_reward = 0.0
        requested_tile_keys = set()
        for request in requests:
            vid = request['video']
            tiles = request['tiles']

            for tile in tiles:
                layer = tile['layer']
                tile_idx = tile['tile']
                tile_key = (vid, layer, tile_idx)
                
                p = float(current_p.get(tile_key, 0.0))
                w = self.prelec_w(p, theta=self.theta)

                prefetch = tile['events']['alpha_M_u']
                
                if prefetch == 1:
                    expected_outcome = self.gain_if_prefetched
                    requested_tile_keys.add(tile_key)
                else:
                    expected_outcome = self.loss_if_not_prefetched
                
                v = self.cpt_value(expected_outcome, lam=self.lam)
                expected_reward += w * v

        return float(expected_reward)

    def prelec_w(self, p: float, theta: float = 0.65) -> float:
        if p <= 0.0: return 0.0
        if p >= 1.0: return 1.0
        return math.exp(-((-math.log(p)) ** theta))

    def cpt_value(
        self,
        x: float, 
        lam: float = 1.5
    ) -> float:
        if x >= 0:
            return x
        else:
            return -lam * (-x)

    def reset(self, **kwargs):
        self.step_count = 0

        _, info_users = self.users_env.reset(**kwargs)
        info_cache_list = [cache.reset(**kwargs)[1] for cache in self.du_caches] if self.du_caches else []
        info_cache = {'du_caches_info': info_cache_list}
        info_cache = {
            **info_cache,
            **(self.mec_cache.reset(**kwargs)[1] if self.mec_cache else {})
        }
        # self.latency_model.reset(**kwargs)
        
        info = {
            **info_users, 
            **info_cache
        }

        return None, info 

In [ ]:
def getTiles(step, user, users_viewport_tiles, n) -> np.ndarray:
    mask = np.zeros(n * n, dtype=int)    
    for tx, ty in users_viewport_tiles[user][step]:
        if 0 <= tx < n and 0 <= ty < n:
            mask[ty * n + tx] = 1
    
    return mask

In [ ]:
if __name__ == "__main__":
    n_episodes = 100
    n_nodes = 3
    n_users = 100
    step_size = 5.0
    alpha = 1.0
    n_gops = 60
    n_layers = 2
    n = 4
    max_capacity = 5000e6  # 5000 MB
    n_videos = 1000

    #### CPT parameters ####
    lam = 3.7183
    theta = 0.5

    users_env = UserTileRequestEvents(
        n_nodes=n_nodes,
        n_users=n_users,
        step_size=step_size,
        n_videos=n_videos,
        n_gops=n_gops,
        n_layers=n_layers,
        n_tiles=n*n,
        n=n,
        alpha=alpha,
        users_viewport_tiles=None,
        requested_videos=None
    )

    # du_caches = [,
    #     CacheEngineEnv(
    #         n_tiles=n*n,
    #         n_videos=n_videos,
    #         cache_capacity=max_capacity
    #     ) for _ in range(n_nodes)  # Number of DUs = n_nodes
    # ]
    du_caches = []

    mec_cache = CacheEngineEnv(
        n_tiles=n*n,
        n_videos=n_videos,
        cache_capacity=max_capacity
    )

    # Create latency model (replace numbers with your real config)
    P = n_nodes; max_U = n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,       # 640 Mbps -> 80e6 B/s 
        R_C_M=1.25e9,     # 10 Gbps -> 1.25e9 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 40e6, dtype=float),    # 320 Mbps -> 40e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,    # 1 ms
        mec_fixed_delay=0.005,   # 5 ms
        cloud_fixed_delay=0.05   # 50 ms
    )

    env = EnvWrapper(
        n=n,
        n_layers=n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        lam=lam,
        theta=theta
    )

    print(
        f"Experiment ===================\n"
        f"Total users: {n_users}\n"
        f"Cache capacity: {max_capacity}MB\n"
        f"Video matrix: {n_videos}x{n_layers}x{n*n}\n"
        f"==============================\n"
    )

    _, info = env.reset()
    viewport_tiles = info['viewport_tiles']
    requested_videos = info['requested_videos']

    done = False
    results = []
    total_reward = 0.0

    for step in count():
        actions = [
            {
                'user': user,
                'video': requested_videos[user],
                'tiles': getTiles(step % n_gops, user, viewport_tiles, n),
                'gop': step % n_gops
            } for user in range(n_users)
        ]

        obs, reward, done, info = env.step(actions)

        total_reward += float(reward)

        results.append({
            "step": step,
            "total_reward": total_reward,
            "cache_hits": info["base_layer_cache_hits"] + info["enh_layer_cache_hits"],
            "cache_misses": info["base_layer_cache_misses"] + info["enh_layer_cache_misses"],
            "info": info
        })

        print(
            f"Step {step} - "
            f"Reward: {reward:.4f} - "
            f"Total Reward: {total_reward:.4f} - "
            f"Cache Hits: {info['base_layer_cache_hits'] + info['enh_layer_cache_hits']} - "
            f"Cache Misses: {info['base_layer_cache_misses'] + info['enh_layer_cache_misses']}\n"
            f"Cache Utilization: {info['cache_utilization']:.2%} - "
            f"Items in Cache: {info['cache_num_items']} - "
            f"Final Capacity: {info['cache_current_capacity']/1e6:.2f}/{info['cache_total_capacity']/1e6:.1f} MB"
        )
        
        if done: 
            break

In [ ]:
if __name__ == "__main__":
    steps = range(1, len(results) + 1)
    cache_hits_series = [r["cache_hits"] for r in results]
    cache_misses_series = [r["cache_misses"] for r in results]

    fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharex=True)

    # Rewards with moving average
    axes[0].plot(steps, [r["total_reward"] for r in results], label="Step reward", alpha=0.7, color="blue")
    w = max(1, min(20, len(results) // 10))
    if w > 1:
        ma = [sum([r["total_reward"] for r in results][i - w:i]) / w for i in range(w, len(results) + 1)]
        axes[0].plot(range(w, len(results) + 1), ma, label=f"Moving avg (w={w})", color="orange")
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Total reward")
    axes[0].set_title("Training Rewards")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # Cache hits
    axes[1].plot(steps, cache_hits_series, label="Cache hits", color="green", alpha=0.8)
    axes[1].set_title("Cache Hits per Step")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Hits")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    # Cache misses
    axes[2].plot(steps, cache_misses_series, label="Cache misses", color="red", alpha=0.8)
    axes[2].set_title("Cache Misses per Step")
    axes[2].set_xlabel("Step")
    axes[2].set_ylabel("Misses")
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
if __name__ == "__main__":
    n_episodes = 100
    n_nodes = 3
    n_users = 100
    step_size = 5.0
    alpha = 1.0
    n_gops = 60
    n_layers = 2
    n = 4
    max_capacity = 5000e6  # 5000 MB
    n_videos = 1000

    #### CPT parameters ####
    lam = 3.7183
    theta = 0.5

    users_env = UserTileRequestEvents(
        n_users=n_users,
        step_size=step_size,
        n_videos=n_videos,
        n_gops=n_gops,
        n_layers=n_layers,
        n_tiles=n*n,
        n=n,
        alpha=alpha,
        users_viewport_tiles=None,
        requested_videos=None
    )

    du_caches = []

    mec_cache = CacheEngineEnv(
        n_tiles=n*n,
        n_videos=n_videos,
        cache_capacity=max_capacity
    )

    # Create latency model (replace numbers with your real config)
    P = n_nodes; max_U = n_users
    lat_model = MultiDULatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,       # 640 Mbps -> 80e6 B/s 
        R_C_M=1.25e9,     # 10 Gbps -> 1.25e9 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 40e6, dtype=float),    # 320 Mbps -> 40e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float), # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,    # 1 ms
        mec_fixed_delay=0.005,   # 5 ms
        cloud_fixed_delay=0.05   # 50 ms
    )

    env = EnvWrapper(
        n=n,
        n_layers=n_layers,
        users_env=users_env, 
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=lat_model,
        lam=lam,
        theta=theta
    )

    print(
        f"Experiment ===================\n"
        f"Total users: {n_users}\n"
        f"Cache capacity: {max_capacity}MB\n"
        f"Video matrix: {n_videos}x{n_layers}x{n*n}\n"
        f"==============================\n"
    )

    _, info = env.reset()
    viewport_tiles = info['viewport_tiles']
    requested_videos = info['requested_videos']

    done = False
    results = []
    total_reward = 0.0

    for step in count():
        actions = [
            {
                'user': user,
                'video': requested_videos[user],
                'tiles': getTiles(step % n_gops, user, viewport_tiles, n),
                'gop': step % n_gops
            } for user in range(n_users)
        ]

        obs, reward, done, info = env.step(actions)

        total_reward += float(reward)

        results.append({
            "step": step,
            "total_reward": total_reward,
            "cache_hits": info["base_layer_cache_hits"] + info["enh_layer_cache_hits"],
            "cache_misses": info["base_layer_cache_misses"] + info["enh_layer_cache_misses"],
            "info": info
        })

        print(
            f"Step {step} - "
            f"Reward: {reward:.4f} - "
            f"Total Reward: {total_reward:.4f} - "
            f"Cache Hits: {info['base_layer_cache_hits'] + info['enh_layer_cache_hits']} - "
            f"Cache Misses: {info['base_layer_cache_misses'] + info['enh_layer_cache_misses']}\n"
            f"Cache Utilization: {info['cache_utilization']:.2%} - "
            f"Items in Cache: {info['cache_num_items']} - "
            f"Final Capacity: {info['cache_current_capacity']/1e6:.2f}/{info['cache_total_capacity']/1e6:.1f} MB"
        )
        
        if done: 
            break